In [3]:
import pandas as pd
import numpy as np

import pandas as pd
import numpy as np

# 1. 파일 불러오기 (r을 붙이고 내 PC의 정확한 전체 경로를 써주기!)
# 예시 경로니까, 실제 파일이 있는 폴더 주소로 싹 바꿔줘!

# 1) 기본 조류 통합 데이터 경로
base_path = r'C:\Users\pc\OneDrive\바탕 화면\유근서\.2026 기후부 ax 아이디어 경진대회\ax 공모전 파이썬\2026-AX-Contest-Water\data\대청호 조류 전처리 완료 통합 데이터.csv'
df_base = pd.read_csv(base_path)

# 2) 수위 통합 데이터 경로 (아까 water_level 폴더에 저장했지?)
dam_path = r'C:\Users\pc\OneDrive\바탕 화면\유근서\.2026 기후부 ax 아이디어 경진대회\ax 공모전 파이썬\2026-AX-Contest-Water\data\water_level\대청댐_수문자료_통합본_2015_2026.csv'
df_dam = pd.read_csv(dam_path)

# 3) TN/TP 통합 데이터 경로 (아까 tntp 폴더에 저장했지?)
nutrients_path = r'C:\Users\pc\OneDrive\바탕 화면\유근서\.2026 기후부 ax 아이디어 경진대회\ax 공모전 파이썬\2026-AX-Contest-Water\data\tntp\대청호_TN_TP_통합본_2015_2026.csv'
df_nutrients = pd.read_csv(nutrients_path)

# 2. 날짜 형식 통일 (병합의 핵심!)
df_base['조사일'] = pd.to_datetime(df_base['조사일'])
df_nutrients['조사일'] = pd.to_datetime(df_nutrients['조사일'])
df_dam['일시'] = pd.to_datetime(df_dam['일시'])

# 3. 1차 병합: 조류 데이터 + 영양염류 (날짜와 채수위치 두 가지가 모두 일치해야 함)
print("🔗 1차 병합 중: 조류 데이터와 TN/TP 데이터를 결합합니다...")
df_merge1 = pd.merge(df_base, df_nutrients, on=['조사일', '채수위치'], how='left')

# 4. 2차 병합: 수문 데이터 결합 (날짜 기준으로 결합)
print("🔗 2차 병합 중: 수문(수위/방류량) 데이터를 결합합니다...")
df_final = pd.merge(df_merge1, df_dam, left_on='조사일', right_on='일시', how='left')

# 5. [차별화 전략] 환경공학적 파생 변수 생성
print("🧪 환경공학적 파생 변수(N/P 비율, 체류시간)를 생성합니다...")

# (1) N/P 비율: 조류 성장의 제한 인자 판단 (TP가 0인 경우 대비해 아주 작은 값 추가)
df_final['N_P_비율'] = df_final['TN(㎎/L)'] / (df_final['TP(㎎/L)'] + 1e-9)

# (2) 체류 시간 지표: 저수량 / 총방류량 (방류량이 0인 경우 대비해 0.1 추가)
df_final['체류시간_지표'] = df_final['저수량(백만㎥)'] / (df_final['총방류량(㎥/s)'] + 0.1)

# 6. 불필요한 중복 컬럼 및 결측치 정리
# 병합 시 생긴 '일시' 컬럼 삭제 및 모델 학습에 방해되는 행 제거
df_final = df_final.drop(columns=['일시'])
df_final = df_final.dropna(subset=['유해남조류 세포수 (cells/㎖)'])

# 7. 결과 확인 및 저장
print(f"\n✅ 최종 통합 완료! 데이터 형태: {df_final.shape}")
print(f"추가된 핵심 변수: ['TN(㎎/L)', 'TP(㎎/L)', 'N_P_비율', '수위(EL.m)', '체류시간_지표']")

df_final.to_csv('대청호_AI_학습용_최종_마스터데이터.csv', index=False, encoding='utf-8-sig')
print("\n💾 '대청호_AI_학습용_최종_마스터데이터.csv'로 저장되었습니다.")

🔗 1차 병합 중: 조류 데이터와 TN/TP 데이터를 결합합니다...
🔗 2차 병합 중: 수문(수위/방류량) 데이터를 결합합니다...
🧪 환경공학적 파생 변수(N/P 비율, 체류시간)를 생성합니다...

✅ 최종 통합 완료! 데이터 형태: (1831, 37)
추가된 핵심 변수: ['TN(㎎/L)', 'TP(㎎/L)', 'N_P_비율', '수위(EL.m)', '체류시간_지표']

💾 '대청호_AI_학습용_최종_마스터데이터.csv'로 저장되었습니다.
